## DLT Pipeline for Data Loading: Raw Data to Bronze, Silver and Gold

This Delta Live Tables (DLT) pipeline ingests raw CSV files and loads them into Bronze Delta tables, preserving the raw data in its original structure while converting it into reliable, versioned Delta format.

The data is then transformed and cleansed in the Silver layer, where schema enforcement, data quality checks, deduplication, and standardization are applied. At this stage, business rules are implemented, invalid records are dropped, and relationships between datasets are established to create structured, analytics-ready tables.

Finally, the refined data is aggregated and modeled in the Gold layer, where it is optimized for business consumption. Gold tables contain curated, high-value datasets such as KPIs, summary tables, and dimensional models that support reporting, dashboards, and advanced analytics use cases. Slowly Changing Dimension (SCD) logic is also applied where required to track historical changes in dimensional data, ensuring accurate and reliable analytical reporting over time.

### Import necessary packages

In [0]:
import dlt
import re
from pyspark.sql.functions import date_format, from_utc_timestamp, current_timestamp, col, year, month, dayofmonth, quarter, weekofyear, dayofweek, when, coalesce, lit, sum as _sum, max as _max, trim, expr
from pyspark.sql.types import DecimalType

### Helper Functions

In [0]:
# Function to check if a file path exists
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

# Function to clean column names
def clean_column_names(name):
    name = name.lower()
    cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', name)
    cleaned_name = re.sub(r'__+', '_', cleaned_name)
    return cleaned_name.rstrip('_')

### Fetching values from config file

In [0]:
# Fetch config file path
config_path = spark.conf.get("config_path")

# Read config file
config = spark.read.option("multiline","true").json(config_path).collect()[0]

# Fetch values
bronze_tables = config["bronze_tables"]
bronze_source_relative_path = config["bronze_source_relative_path"]

# Define current timestamp
current_ts = date_format(from_utc_timestamp(current_timestamp(), "Asia/Kolkata"), "yyyy-MM-dd HH:mm:ss").cast("date")

# Set decimal type
decimal_type = DecimalType(18,3)


### Bronze Tables

All raw CSV files are ingested into their respective Bronze Delta tables using a single reusable function combined with a for loop. This approach makes the pipeline efficient, scalable, and easy to maintain, as the same logic is applied consistently across all datasets. By dynamically creating tables for each available file, the process remains generic and flexible, allowing new data sources to be added with minimal effort.


In [0]:
def create_bronze_table(table_name, source_path):

    @dlt.table(
        name=f"bronze.{table_name}",
        comment="Raw data ingested from CSV file",
        table_properties={
            "quality": "bronze"
        }
    )
    def bronze_table():
        
        # Read the data from raw CSV file 
        bronze_df = (spark.read
            .format("csv")
            .option("header", "true")
            .load(source_path)
        )
        
        # Clean column names
        for col_name in bronze_df.columns:
            cleaned_name = clean_column_names(col_name)
            bronze_df = bronze_df.withColumnRenamed(col_name, cleaned_name)

        # Add audit column
        bronze_df = bronze_df.withColumn("dw_load_ts", current_ts)

        return bronze_df

# Loop through all bronze tables
for bronze_table in bronze_tables:

    bronze_source_path = f"{bronze_source_relative_path}/{bronze_table}.csv"

    if path_exists(bronze_source_path):
        create_bronze_table(bronze_table, bronze_source_path)

### Silver Tables

In [0]:
@dlt.table(
    name="silver.customer",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_market", "market is not null")

def silver_customer():
    customer_df = dlt.read("bronze.master_customer").select(
        col("customer_id").alias("market"),
        col("customer_description").alias("market_description"),
        col("customer_region").alias("market_region"),
        col("customer_country_region").alias("market_sub_region"),
        col("demand_type"),
        col("demand_sub_type"),
        col("latitude"),
        col("longttitude")
    )

    # Add audit column
    customer_df = customer_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return customer_df

In [0]:
@dlt.table(
    name="silver.customer_product",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_market", "market is not null")
@dlt.expect_or_drop("valid_material", "material is not null")

def silver_customer_product():
    customer_product_df = dlt.read("bronze.master_customer_product").select(
        col("customer_id").alias("market"),
        col("product_id").alias("material"),
        col("market_segment").alias("market_segment"),
        col("remaining_shelf_life")
    )

    # Add audit column
    customer_product_df = customer_product_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return customer_product_df

In [0]:
@dlt.table(
    name="silver.location",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_plant", "plant is not null")

def silver_location():
    location_df = dlt.read("bronze.master_location").select(
        col("location_id").alias("plant"),
        col("location_description").alias("plant_description"),
        col("location_region").alias("plant_region"),
        col("location_type").alias("plant_type"),
        col("latitude"),
        col("longttitude")
    )

    # Add audit column
    location_df = location_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return location_df

In [0]:
@dlt.table(
    name="silver.location_product",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_plant", "plant is not null")
@dlt.expect_or_drop("valid_material", "material is not null")

def silver_location_product():
    location_product_df = dlt.read("bronze.master_location_product").select(
        col("location_id").alias("plant"),
        col("product_id").alias("material"),
        col("material_description").alias("material_description"),
        col("plant_exclusion_x"),
        col("market_dc_x"),
    )

    # Add audit column
    location_product_df = location_product_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return location_product_df

In [0]:
@dlt.table(
    name="silver.product",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_material", "material is not null")

def silver_product():
    mara_df = dlt.read("bronze.mara").alias("mara")
    makt_df = dlt.read("bronze.makt").alias("makt")

    product_df = (
        mara_df
        .join(
            makt_df,
            (col("mara.matnr") == col("makt.matnr")) & (col("makt.spras") == "E"),
            "left"
        )
        .select(
            col("mara.matnr").alias("material"),
            col("makt.maktx").alias("material_description"),
            col("makt.maktg").alias("material_description_2"),
            col("mara.mtart").alias("material_type"),
            col("mara.matkl").alias("material_group"),
            col("mara.meins").alias("base_unit_of_measure"),
            col("mara.vpsta").alias("complete_status"),
            col("mara.pstat").alias("maintenance_status"),
            col("mara.mstae").alias("xplant_status")
        )
    )

    # Drop duplicates
    product_df = product_df.dropDuplicates(["material"])

    # Add audit column
    product_df = product_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return product_df

In [0]:
@dlt.table(
    name="silver.batch",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_batch", "batch is not null")

def silver_batch():
    mch1_df = dlt.read("bronze.mch1").alias("mch1")
    lfa1_df = dlt.read("bronze.lfa1").alias("lfa1")

    batch_df = (
        mch1_df
        .join(
            lfa1_df,
            col("mch1.lifnr") == col("lfa1.lifnr"),
            "left"
        )
        .select(
            col("mch1.matnr").alias("material"),
            col("mch1.charg").alias("batch"),
            col("mch1.vfdat").alias("expiration_date"),
            col("mch1.hsdat").alias("manufacturing_date"),
            col("mch1.qndat").alias("next_inspection_date"),
            col("mch1.lwedt").alias("last_goods_receipt_date"),
            col("mch1.licha").alias("supplier_batch"),
            col("mch1.lifnr").alias("supplier_number"),
            col("lfa1.name1").alias("supplier_name"),
            col("mch1.ersda").alias("created_on"),
            col("mch1.ernam").alias("created_by"),
            col("mch1.laeda").alias("changed_on"),
            col("mch1.aenam").alias("changed_by")
        )
    )

    # Drop duplicates
    batch_df = batch_df.dropDuplicates(["material", "batch"])

    # Add audit column
    batch_df = batch_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return batch_df

In [0]:
@dlt.table(
    name="silver.supplier",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_supplier_number", "supplier_number is not null")

def silver_supplier():
    lfm1_df = dlt.read("bronze.lfm1").alias("lfm1")
    lfa1_df = dlt.read("bronze.lfa1").alias("lfa1")
    lfb1_df = dlt.read("bronze.lfb1").alias("lfb1")

    supplier_df = (
        lfm1_df
        .join(
            lfa1_df,
            col("lfm1.lifnr") == col("lfa1.lifnr"),
            "left"
        )
        .join(
            lfb1_df,
            col("lfm1.lifnr") == col("lfb1.lifnr"),
            "left"
        )
        .select(
            col("lfm1.lifnr").alias("supplier_number"),
            col("lfm1.ekorg").alias("purchasing_org"),
            col("lfm1.waers").alias("order_currency"),
            col("lfm1.zterm").alias("terms_of_payment"),
            col("lfm1.inco1").alias("incoterms"),
            col("lfa1.land1").alias("country_region_key"),
            col("lfa1.name1").alias("supplier_name"),
            col("lfa1.ort01").alias("city"),
            col("lfa1.pstlz").alias("postal_code"),
            col("lfa1.pstl2").alias("po_box"),
            col("lfa1.stras").alias("street"),
            col("lfa1.adrnr").alias("adress"),
            col("lfa1.erdat").alias("created_on"),
            col("lfa1.ernam").alias("created_by"),
            col("lfa1.ktokk").alias("account_group"),
            col("lfb1.bukrs").alias("company_code"),
            col("lfb1.intad").alias("supplier_email")
        )
    )

    # Drop duplicates
    supplier_df = supplier_df.dropDuplicates(["supplier_number"])

    # Add audit column
    supplier_df = supplier_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return supplier_df

In [0]:
@dlt.table(
    name="silver.uom",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_uom", "uom is not null")

def silver_uom():
    uom_df = dlt.read("bronze.mara").select(
        col("meins").alias("uom")
    ).distinct()

    # Add audit column
    uom_df = uom_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return uom_df

In [0]:
@dlt.table(
    name="silver.storage",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_plant", "plant is not null")

def silver_storage():
    storage_df = dlt.read("bronze.t001l").select(
        col("werks").alias("plant"),
        col("lgort").alias("storage_location"),
        col("lgobe").alias("storage_description")
    ).distinct()

    # Add audit column
    storage_df = storage_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return storage_df

In [0]:
@dlt.table(
    name="silver.currency",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_currency", "currency is not null")

def silver_currency():
    currency_df = dlt.read("bronze.tcurf").select(
        col("fcurr").alias("currency")
    ).distinct()

    # Add audit column
    currency_df = currency_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return currency_df

In [0]:
@dlt.table(
    name="silver.demand_actual",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_period_id", "period_id is not null")
@dlt.expect_or_drop("valid_material", "material is not null")
@dlt.expect_or_drop("valid_market", "market is not null")

def silver_demand_actual():
    demand_actual_df = dlt.read("bronze.demand_actual").select(
        col("period"),
        col("period_id"),
        col("material"),
        col("market"),
        col("actual_qty"),
        col("actual_revenue"),
        col("actual_orders"),
        col("shipped_qty"),
        col("delivered_qty"),
        col("returns_qty"),
        col("net_actual_qty"),
        col("net_revenue"),
        col("discounts"),
        col("final_revenue"),
        col("service_level")
    )

    # Add audit column
    demand_actual_df = demand_actual_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return demand_actual_df

In [0]:
@dlt.table(
    name="silver.demand_forecast",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_period_id", "period_id is not null")
@dlt.expect_or_drop("valid_material", "material is not null")
@dlt.expect_or_drop("valid_market", "market is not null")

def silver_demand_forecast():
    demand_forecast_df = dlt.read("bronze.demand_forecast").select(
        col("period"),
        col("period_id"),
        col("material"),
        col("market"),
        col("consensus_demand"),
        col("ibp_consensus_forecast"),
        col("budget_volumes"),
        col("sc_forecast_override"),
        col("statistical_forecast"),
        col("forecast_selector"),
        col("upside"),
        col("fcst_snapshot_1"),
        col("fcst_snapshot_2"),
        col("fcst_snapshot_3"),
        col("fcst_snapshot_4"),
        col("fcst_snapshot_5"),
        col("fcst_snapshot_6"),
        col("fcst_snapshot_7"),
        col("fcst_snapshot_8"),
        col("fcst_snapshot_9"),
        col("fcst_snapshot_10"),
        col("fcst_snapshot_11"),
        col("fcst_snapshot_12"),
        col("ibp_forecast"),
        col("fcst_variance_snapshot_1"),
        col("fcst_variance_snapshot_2"),
        col("fcst_variance_snapshot_3"),
        col("fcst_variance_perc_snapshot_1"),
        col("fcst_variance_perc_snapshot_2"),
        col("fcst_variance_perc_snapshot_3"),
        col("ibp_consensus_forecast_prior")
    )

    # Add audit column
    demand_forecast_df = demand_forecast_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return demand_forecast_df

In [0]:
@dlt.table(
    name="silver.batch_release_external",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_inspection_lot", "inspection_lot is not null")

def silver_batch_release_external():
    qals_df = dlt.read("bronze.qals").alias("qals")
    t001w_df = dlt.read("bronze.t001w").alias("t001w")
    tq30t_df = dlt.read("bronze.tq30t").alias("tq30t")
    tq31t_df = dlt.read("bronze.tq31t").alias("tq31t")
    t001l_df = dlt.read("bronze.t001l").alias("t001l")
    lfa1_df = dlt.read("bronze.lfa1").alias("lfa1")
    mara_df = dlt.read("bronze.mara").alias("mara")
    t023t_df = dlt.read("bronze.t023t").alias("t023t")
    t134t_df = dlt.read("bronze.t134t").alias("t134t")
    mch1_df = dlt.read("bronze.mch1").alias("mch1")
    marc_df = dlt.read("bronze.marc").alias("marc")
    qave_df = dlt.read("bronze.qave").alias("qave")

    batch_release_external_df = (
        qals_df
        .join(
            t001w_df, 
            col("qals.werk") == col("t001w.werks"), 
            "left"
        )
        .join(
            tq30t_df, 
            (col("qals.art") == col("tq30t.art")) & (col("tq30t.sprache") == 'E'), 
            "left"
        )
        .join(
            tq31t_df, 
            (col("qals.herkunft") == col("tq31t.herkunft")) & (col("tq31t.sprache") == 'E'), 
            "left"
        )
        .join(
            t001l_df, 
            col("qals.werk") == col("t001l.werks"), 
            "left"
        )
        .join(
            mara_df, 
            col("qals.matnr") == col("mara.matnr"), 
            "left"
        )
        .join(
            t023t_df, 
            (col("mara.matkl") == col("t023t.matkl")) & (col("t023t.spras") == 'E'), 
            "left"
        )
        .join(
            t134t_df, 
            (col("mara.mtart") == col("t134t.mtart")) & (col("t134t.spras") == 'E'), 
            "left"
        )
        .join(
            mch1_df, 
            (col("qals.matnr") == col("mch1.matnr")) & (col("qals.charg") == col("mch1.charg")), 
            "left"
        )
        .join(
            marc_df, 
            (col("qals.werk") == col("marc.werks")) & (col("qals.matnr") == col("marc.matnr")), 
            "left"
        )
        .join(
            qave_df, 
            (col("qals.prueflos") == col("qave.prueflos")) & (col("qals.werk") == col("qave.vwerks")) & (col("qave.vbewertung") == "L"), 
            "left"
        )
        .select(
            col("qals.prueflos").alias("inspection_lot"),
            col("qals.werk").alias("plant"),
            col("t001w.name1").alias("plant_desc"),
            col("qals.art").alias("inspection_type"),
            col("qals.herkunft").alias("inspection_type_origin"),
            col("tq31t.herktxt").alias("inspection_type_origin_desc"),
            col("t001l.lgort").alias("storage_location"),
            col("t001l.lgobe").alias("storage_location_desc"),
            col("qals.charg").alias("batch"),
            col("qals.lichn").alias("supplier_batch"),
            col("mara.matkl").alias("material_group"),
            col("t023t.wgbez60").alias("material_group_description"),
            col("mara.mtart").alias("material_type"),
            col("t134t.mtbez").alias("material_type_description"),
            col("mch1.ersda").alias("batch_creation_date"),
            col("mch1.vfdat").alias("shelf_life_expiration_date"),
            col("mch1.hsdat").alias("date_of_manufacture"),
            col("qals.ersteldat").alias("inspection_lot_created_on"),
            col("qals.aenderdat").alias("inspection_lot_changed_on"),
            col("marc.webaz").alias("planned_gr_processing_time_in_days"),
            col("qals.pastrterm").alias("scheduled_start_date"),
            col("qals.paendterm").alias("scheduled_end_date"),
            col("qals.losmenge").alias("inspection_lot_qty"),
            col("qals.lmengeist").alias("actual_lot_qty"),
            col("qals.lmengepr").alias("inspected_qty"),
            col("qals.lmengesch").alias("defective_qty")
        )
    )

    # Drop duplicates
    batch_release_external_df = batch_release_external_df.dropDuplicates(["inspection_lot"])

    # Add audit column
    batch_release_external_df = batch_release_external_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return batch_release_external_df

In [0]:
@dlt.table(
    name="silver.batch_release_internal",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_inspection_lot", "inspection_lot is not null")

def silver_batch_release_internal(): 
    qals_df = dlt.read("bronze.qals").alias("qals")
    t001w_df = dlt.read("bronze.t001w").alias("t001w")
    tq30t_df = dlt.read("bronze.tq30t").alias("tq30t")
    tq31t_df = dlt.read("bronze.tq31t").alias("tq31t")
    t001l_df = dlt.read("bronze.t001l").alias("t001l")
    qave_df = dlt.read("bronze.qave").alias("qave")

    batch_release_internal_df = (
        qals_df
        .join(
            t001w_df, 
            col("qals.werk") == col("t001w.werks"), 
            "left"
        )
        .join(
            tq30t_df, 
            (col("qals.art") == col("tq30t.art")) & (col("tq30t.sprache") == 'E'), 
            "left"
        )
        .join(
            tq31t_df, 
            (col("qals.herkunft") == col("tq31t.herkunft")) & (col("tq31t.sprache") == 'E'), 
            "left"
        )
        .join(
            t001l_df, 
            col("qals.werk") == col("t001l.werks"), 
            "left"
        )
        .join(
            qave_df, 
            (col("qals.prueflos") == col("qave.prueflos")) & (col("qals.werk") == col("qave.vwerks")) & (col("qave.vbewertung") == "L"), 
            "left"
        )
    ).select(
        col("qals.prueflos").alias("inspection_lot"),
        col("qals.werk").alias("plant"),
        col("t001w.name1").alias("plant_desc"),
        col("qals.art").alias("inspection_type"),
        col("tq30t.kurztext").alias("inspection_type_desc"),
        col("qals.herkunft").alias("inspection_type_origin"),
        col("tq31t.herktxt").alias("inspection_type_origin_desc"),
        col("t001l.lgobe").alias("storage_location_desc"),
        col("qals.ersteldat").alias("inspection_lot_created_on"),
        col("qals.aenderdat").alias("inspection_lot_changed_on"),
        col("qals.pastrterm").alias("scheduled_start_date"),
        col("qals.paendterm").alias("scheduled_end_date"),
        col("qals.losmenge").alias("inspection_lot_qty"),
        col("qals.lmengeist").alias("actual_lot_qty"),
        col("qals.lmengepr").alias("inspected_qty"),
        col("qals.lmengesch").alias("defective_qty"),
        col("qave.vcode").alias("ud_code"),
        col("qave.vname").alias("ud_made_by"),
        col("qave.vdatum").alias("ud_made_on")
    )

    # Drop duplicates
    batch_release_internal_df = batch_release_internal_df.dropDuplicates(["inspection_lot"])

    # Add audit column
    batch_release_internal_df = batch_release_internal_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return batch_release_internal_df

In [0]:
@dlt.table(
    name="silver.purchase_order",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_purchase_order", "purchase_order is not null")
@dlt.expect_or_drop("valid_item", "item is not null")

def silver_purchase_order():     
    ekpo_df = dlt.read("bronze.ekpo").alias("ekpo")     # po item
    ekko_df = dlt.read("bronze.ekko").alias("ekko")     # po header
    eket_df = dlt.read("bronze.eket").alias("eket")     # schedule lines
    t001_df = dlt.read("bronze.t001").alias("t001")     # company
    t001w_df = dlt.read("bronze.t001w").alias("t001w")  # plant
    lfa1_df = dlt.read("bronze.lfa1").alias("lfa1")     # vendor
    t023t_df = dlt.read("bronze.t023t").filter(col("spras") == "e").alias("t023t")  # material group text
    makt_df = dlt.read("bronze.makt").filter(col("spras") == "e").alias("makt")     # material text
    t024_df = dlt.read("bronze.t024").alias("t024")     # purchasing group
    t024e_df = dlt.read("bronze.t024e").alias("t024e")  # purchasing org text

    eket_agg_df = (
        eket_df.groupby("ebeln", "ebelp")
        .agg(
            _sum("menge").alias("schedule_qty"),
            _sum("wemng").alias("delivered_qty"),
            _max("eindt").alias("next_schedule_date")
        )
        .alias("eket")
    )

    po_joined_df = (
        ekpo_df
        .join(
            ekko_df,
            col("ekpo.ebeln") == col("ekko.ebeln"),
            "left"
        )
        .join(
            eket_agg_df,
            (col("ekpo.ebeln") == col("eket.ebeln")) &
            (col("ekpo.ebelp") == col("eket.ebelp")),
            "left"
        )
        .join(
            t001_df,
            col("ekpo.bukrs") == col("t001.bukrs"),
            "left"
        )
        .join(
            t001w_df.alias("plant"),
            col("ekpo.werks") == col("plant.werks"),
            "left"
        )
        .join(
            lfa1_df,
            col("ekko.lifnr") == col("lfa1.lifnr"),
            "left"
        )
        .join(
            t023t_df,
            col("ekpo.matkl") == col("t023t.matkl"),
            "left"
        )
        .join(
            makt_df,
            col("ekpo.matnr") == col("makt.matnr"),
            "left"
        )
        .join(
            t024_df,
            col("ekko.ekgrp") == col("t024.ekgrp"),
            "left"
        )
        .join(
            t024e_df,
            col("ekko.ekorg") == col("t024e.ekorg"),
            "left"
        )
    )

    order_qty = col("ekpo.menge")
    delivered_qty = coalesce(col("eket.delivered_qty"), lit(0))
    still_to_deliver = order_qty - delivered_qty
    qty_to_invoice = still_to_deliver
    value_to_be_delivered = still_to_deliver * col("ekpo.netpr")
    value_to_be_invoiced = qty_to_invoice * col("ekpo.netpr")
    po_status = when(still_to_deliver > 0, "open").otherwise("closed")
    po_type = (
        when(col("ekko.bsart") == "nb", "direct")
        .when(col("ekko.bsart") == "zarb", "indirect")
        .when(col("ekko.bsart") == "ub", "sto")
        .otherwise("other")
    )
    po_sub_type = (
        when(trim(col("ekpo.pstyp")) == "l", "subcontract po")
        .when(
            col("ekpo.pstyp").isNull() |
            (trim(col("ekpo.pstyp")) == ""),
            "turn key po"
        )
        .otherwise("other")
    )
    sourcing_type = (
        when(col("ekko.bsart") == "UB", "Internal Mfg")
        .otherwise("External Mfg")
    )

    purchase_order_df = po_joined_df.select(
        col("ekpo.werks").alias("plant"),
        col("plant.name1").alias("plant_name"),
        col("ekpo.ebeln").alias("purchase_order"),
        col("ekpo.ebelp").alias("item"),
        col("ekpo.ebelp").alias("purchase_order_item"),
        col("ekpo.matnr").alias("material"),
        col("makt.maktx").alias("material_name"),
        col("ekpo.matkl").alias("material_group"),
        col("t023t.wgbez").alias("material_group_description"),
        order_qty.alias("order_quantity"),
        col("eket.schedule_qty").alias("quantity_to_be_delivered"),
        delivered_qty.alias("delivered_quantity"),
        qty_to_invoice.alias("quantity_to_be_invoiced"),
        col("eket.next_schedule_date").alias("next_schedule_line_date"),
        col("ekpo.netpr").alias("net_order_price"),
        col("ekko.ktwrt").alias("net_order_value"),
        value_to_be_delivered.alias("value_to_be_delivered"),
        value_to_be_invoiced.alias("value_to_be_invoiced"),
        col("ekko.waers").alias("order_currency"),
        col("ekko.bedat").alias("order_date"),
        col("ekko.statu").alias("order_status"),
        po_status.alias("po_status"),
        col("ekko.bsart").alias("order_type"),
        po_type.alias("po_type"),
        po_sub_type.alias("po_sub_type"),
        sourcing_type.alias("sourcing_type"),
        col("ekpo.meins").alias("order_unit"),
        col("ekpo.meins").alias("price_unit"),
        col("ekko.ekgrp").alias("purchasing_group"),
        col("t024.eknam").alias("purchasing_group_name"),
        col("ekko.ekorg").alias("purchasing_organization"),
        col("t024e.ekotx").alias("purchasing_organization_name"),
        lit(None).cast("string").alias("purchasing_info_record"),
        col("ekpo.lgort").alias("storage_location"),
        col("lfa1.lifnr").alias("supplier_id"),
        col("lfa1.name1").alias("supplier_name"),
        lit(None).cast("string").alias("supplying_plant"),
        lit(None).cast("string").alias("supplying_plant_name"),
        col("ekko.ernam").alias("created_by"),
        col("ekko.aedat").alias("created_on"),
        col("t001.butxt").alias("company_name"),
        col("ekpo.bukrs").alias("company_code")
    )

    # Drop duplicates
    purchase_order_df = purchase_order_df.dropDuplicates(["purchase_order","item"])

    # Add audit column
    purchase_order_df = purchase_order_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return purchase_order_df

In [0]:
@dlt.table(
    name="silver.inventory",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_material", "material is not null")
@dlt.expect_or_drop("valid_plant", "plant is not null")
@dlt.expect_or_drop("valid_storage_location", "storage_location is not null")
@dlt.expect_or_drop("valid_batch", "batch is not null")

def silver_inventory():
    mchb_df = dlt.read("bronze.mchb").alias("mchb")
    mch1_df = dlt.read("bronze.mch1").alias("mch1")
    mara_df = dlt.read("bronze.mara").alias("mara")
    makt_df = dlt.read("bronze.makt").alias("makt")
    marc_df = dlt.read("bronze.marc").alias("marc")
    t001w_df = dlt.read("bronze.t001w").alias("t001w")
    t001_df = dlt.read("bronze.t001").alias("t001")
    lfa1_df = dlt.read("bronze.lfa1").alias("lfa1")
    t001k_df = dlt.read("bronze.t001k").alias("t001k")

    inventory_df = (
        mchb_df
        .join(
            mara_df, 
            col("mchb.matnr") == col("mara.matnr"), 
            "left"
        )
        .join(
            makt_df, 
            col("mchb.matnr") == col("makt.matnr"), 
            "left"
        )
        .join(
            mch1_df,
            (col("mchb.matnr") == col("mch1.matnr")) & (col("mchb.charg") == col("mch1.charg")),
            "left"
        )
        .join(
            lfa1_df, 
            col("mch1.lifnr") == col("lfa1.lifnr"), 
            "left"
        )
        .join(
            t001w_df, 
            col("mchb.werks") == col("t001w.werks"), 
            "left"
        )
        .join(
            marc_df,
            (col("mchb.matnr") == col("marc.matnr")) & (col("mchb.werks") == col("marc.werks")),
            "left"
        )
        .join(
            t001k_df, 
            col("t001w.bwkey") == col("t001k.bwkey"), 
            "left"
        )
        .join(
            t001_df, 
            col("t001k.bukrs") == col("t001.bukrs"), 
            "left"
        )
        .select(
            col("mchb.matnr").alias("material"),
            col("mchb.werks").alias("plant"),
            col("mchb.lgort").alias("storage_location"),
            col("mchb.charg").alias("batch"),
            col("makt.maktx").alias("material_description"),
            col("mara.bismt").alias("old_material_number"),
            col("mara.matkl").alias("product_group"),
            col("mara.mtart").alias("product_type"),
            col("t001w.name1").alias("plant_description"),
            col("t001.waers").alias("display_currency"),
            col("mch1.licha").alias("supplier_batch"),
            col("mch1.lifnr").alias("supplier"),
            col("lfa1.name1").alias("supplier_name"),
            col("mch1.hsdat").alias("manufacture_date"),
            col("mch1.vfdat").alias("shelf_life_expiration_date"),
            col("marc.plifz").alias("mrp_lead_time"),
            col("marc.webaz").alias("gr_processing_time"),
            col("marc.eisbe").alias("safety_stock"),
            col("mchb.clabs").cast(decimal_type).alias("unrestricted_qty"),
            col("mchb.cinsm").cast(decimal_type).alias("quality_inspection_qty"),
            col("mchb.cspem").cast(decimal_type).alias("blocked_qty"),
            (
                coalesce(col("mchb.clabs").cast(decimal_type), lit(0)) +
                coalesce(col("mchb.cinsm").cast(decimal_type), lit(0))
            ).alias("available_stock_qty"),
            (
                coalesce(col("mchb.clabs").cast(decimal_type), lit(0)) +
                coalesce(col("mchb.cinsm").cast(decimal_type), lit(0)) +
                coalesce(col("mchb.cspem").cast(decimal_type), lit(0))
            ).alias("total_stock_qty")
        )
    )

    # Drop duplicates
    inventory_df = inventory_df.dropDuplicates(["material", "plant", "storage_location", "batch"])

    # Add audit column
    inventory_df = inventory_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return inventory_df

In [0]:
@dlt.table(
    name="silver.inventory_month_end_stock",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_material", "material is not null")
@dlt.expect_or_drop("valid_plant", "plant is not null")
@dlt.expect_or_drop("valid_period_id", "period_id is not null")

def silver_inventory_month_end_stock():
    mchbh_df = dlt.read("bronze.mchbh").alias("mchbh")   # Batch history
    mardh_df = dlt.read("bronze.mardh").alias("mardh")   # Storage location history
    march_df = dlt.read("bronze.march").alias("march")   # Transit history
    marc_df = dlt.read("bronze.marc").alias("marc")      # Batch indicator
    mbew_df = dlt.read("bronze.mbew").alias("mbew")      # Material valuation

    batch_stock_df = (
        mchbh_df
        .join(
            marc_df,
            (col("mchbh.matnr") == col("marc.matnr")) & (col("mchbh.werks") == col("marc.werks")),
            "inner"
        )
        .filter(col("marc.xchar") == "X")
        .select(
            col("mchbh.matnr"),
            col("mchbh.werks"),
            col("mchbh.lgort"),
            col("mchbh.charg"),
            col("mchbh.lfgja").alias("year"),
            col("mchbh.lfmon").alias("month"),
            expr("try_cast(mchbh.clabs as decimal(18,3))").alias("unrestricted_qty"),
            expr("try_cast(mchbh.cinsm as decimal(18,3))").alias("quality_qty"),
            expr("try_cast(mchbh.cspem as decimal(18,3))").alias("blocked_qty"),
            lit(0).cast(decimal_type).alias("transit_qty")
        )
    )

    non_batch_stock_df = (
        mardh_df
        .join(
            marc_df,
            (col("mardh.matnr") == col("marc.matnr")) & (col("mardh.werks") == col("marc.werks")),
            "left"
        )
        .filter(
            (col("marc.xchar").isNull()) | (col("marc.xchar") != "X")
        )
        .select(
            col("mardh.matnr"),
            col("mardh.werks"),
            col("mardh.lgort"),
            lit(None).alias("charg"),
            col("mardh.lfgja").alias("year"),
            col("mardh.lfmon").alias("month"),
            expr("try_cast(mardh.labst as decimal(18,3))").alias("unrestricted_qty"),
            expr("try_cast(mardh.insme as decimal(18,3))").alias("quality_qty"),
            expr("try_cast(mardh.speme as decimal(18,3))").alias("blocked_qty"),
            lit(0).cast(decimal_type).alias("transit_qty")
        )
    )

    transit_stock_df = (
        march_df
        .select(
            col("march.matnr"),
            col("march.werks"),
            lit(None).alias("lgort"),
            lit(None).alias("charg"),
            col("march.lfgja").alias("year"),
            col("march.lfmon").alias("month"),
            lit(0).cast(decimal_type).alias("unrestricted_qty"),
            lit(0).cast(decimal_type).alias("quality_qty"),
            lit(0).cast(decimal_type).alias("blocked_qty"),
            expr("try_cast(march.trame as decimal(18,3))").alias("transit_qty")
        )
    )

    inventory_df = (
        batch_stock_df
        .unionByName(non_batch_stock_df)
        .unionByName(transit_stock_df)
    )

    mbew_df = spark.table("catalog.bronze.mbew").alias("mbew")

    cost_df = (
        mbew_df
        .select(
            col("matnr"),
            col("bwkey").alias("werks"),
            when(
                col("vprsv")=="S",
                expr("try_cast(stprs as decimal(18,3))")
            )
            .otherwise(
                expr("try_cast(verpr as decimal(18,3))")
            ).alias("cost")

        )
    )

    inventory_month_end_stock_df = (
        inventory_df.alias("inv")
        .join(
            cost_df.alias("cost"),
            (
                (col("inv.matnr") == col("cost.matnr")) &
                (col("inv.werks") == col("cost.werks"))
            ),
            "left"
        )
    )

    inventory_month_end_stock_df = (
        inventory_month_end_stock_df
        .withColumn(
            "unrestricted_value",
            col("unrestricted_qty") * col("cost")
        )
        .withColumn(
            "quality_value",
            col("quality_qty") * col("cost")
        )
        .withColumn(
            "transit_value",
            col("transit_qty") * col("cost")
        )
        .withColumn(
            "blocked_value",
            col("blocked_qty") * col("cost")
        )
    )

    inventory_month_end_stock_df = (
        inventory_month_end_stock_df
        .withColumn(
            "available_stock_qty",
            coalesce(col("unrestricted_qty"), lit(0)) +
            coalesce(col("quality_qty"), lit(0)) +
            coalesce(col("transit_qty"), lit(0))
        )
        .withColumn(
            "total_stock_qty",
            col("available_stock_qty") +
            coalesce(col("blocked_qty"), lit(0))
        )
        .withColumn(
            "available_stock_value",
            coalesce(col("unrestricted_value"), lit(0)) +
            coalesce(col("quality_value"), lit(0)) +
            coalesce(col("transit_value"), lit(0))
        )
        .withColumn(
            "total_stock_value",
            col("available_stock_value") +
            coalesce(col("blocked_value"), lit(0))
        )
    )

    inventory_month_end_stock_df = inventory_month_end_stock_df.select(
        col("inv.matnr").alias("material"),
        col("inv.werks").alias("plant"),
        col("inv.lgort").alias("storage_location"),
        col("inv.charg").alias("batch"),
        col("year"),
        col("month"),
        (col("year") * 100 + col("month")).alias("period_id"),
        col("unrestricted_qty"),
        col("quality_qty"),
        col("blocked_qty"),
        col("transit_qty"),
        col("available_stock_qty"),
        col("total_stock_qty"),
        col("cost").alias("unit_cost"),
        col("unrestricted_value"),
        col("quality_value"),
        col("blocked_value"),
        col("transit_value"),
        col("available_stock_value"),
        col("total_stock_value")
    )

    # Drop duplicates
    inventory_month_end_stock_df = inventory_month_end_stock_df.dropDuplicates(["material", "plant", "storage_location", "batch", "period_id"])

    # Add audit column
    inventory_month_end_stock_df = inventory_month_end_stock_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return inventory_month_end_stock_df

### Gold Tables

#### Dimensions

In [0]:
@dlt.table(
    name="gold.dim_date",
    table_properties={
        "quality": "gold"
    }
)
def gold_dim_date():
    # Generate from existing date range
    date_df = spark.sql("""
        SELECT explode(sequence(
            to_date('2015-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )) AS calendar_date
    """).select(
            col("calendar_date"),
            year("calendar_date").alias("year"),
            quarter("calendar_date").alias("quarter"),
            month("calendar_date").alias("month"),
            weekofyear("calendar_date").alias("week"),
            dayofmonth("calendar_date").alias("day"),
            dayofweek("calendar_date").alias("day_of_week"),
            date_format("calendar_date", "MMMM").alias("month_name"),
            date_format("calendar_date", "EEEE").alias("day_name")
        )

    # Add audit column
    date_df = date_df.withColumn("dw_load_ts", current_ts)

    # Return the dataframe
    return date_df

In [0]:
dlt.create_target_table(name="gold.dim_product",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_product", 
  source = "silver.product", 
  keys = ["material"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_customer",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_customer", 
  source = "silver.customer", 
  keys = ["market"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_customer_product",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_customer_product", 
  source = "silver.customer_product", 
  keys = ["market","material"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_location",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_location", 
  source = "silver.location", 
  keys = ["plant"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_location_product",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_location_product", 
  source = "silver.location_product", 
  keys = ["plant","material"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_batch",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_batch", 
  source = "silver.batch", 
  keys = ["batch"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_supplier",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_supplier", 
  source = "silver.supplier", 
  keys = ["supplier_number"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_uom",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_uom", 
  source = "silver.uom", 
  keys = ["uom"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_storage",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_storage", 
  source = "silver.storage", 
  keys = ["plant","storage_location"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.dim_currency",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.dim_currency", 
  source = "silver.currency", 
  keys = ["currency"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

#### Facts

In [0]:
dlt.create_target_table(name="gold.fact_batch_release_external",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_batch_release_external", 
  source = "silver.batch_release_external", 
  keys = ["inspection_lot"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_batch_release_internal",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_batch_release_internal", 
  source = "silver.batch_release_internal", 
  keys = ["inspection_lot"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_inventory",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_inventory", 
  source = "silver.inventory", 
  keys = ["material", "plant", "storage_location", "batch"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_inventory_month_end_stock",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_inventory_month_end_stock", 
  source = "silver.inventory_month_end_stock", 
  keys = ["material", "plant", "storage_location", "batch", "period_id"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_purchase_order",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_purchase_order", 
  source = "silver.purchase_order", 
  keys = ["purchase_order", "item"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_demand_actual",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_demand_actual", 
  source = "silver.demand_actual", 
  keys = ["period_id", "material", "market"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)

In [0]:
dlt.create_target_table(name="gold.fact_demand_forecast",
  table_properties={
    "quality": "gold"
  }
)

dlt.apply_changes(
  target = "gold.fact_demand_forecast", 
  source = "silver.demand_forecast", 
  keys = ["period_id", "material", "market"], 
  sequence_by = col("dw_load_ts"), 
  stored_as_scd_type=1
)